# 9.1 - When to Fine-Tune: The Decision Framework

**Module 9 - Fine-Tuning** | Netsetos GenAI Engineering

This notebook turns the "should I fine-tune?" decision into runnable code: a gap classifier, an escalation ladder, a break-even cost model, a method-picker, a data-quality demo, a PEFT VRAM calculator, a catastrophic-forgetting gate, and an Indic-tokenizer bill comparison. Every cell is a tiny self-contained function you can re-run with your own numbers - there are no model calls here, just the arithmetic and logic that decide when fine-tuning is worth it.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

The only third-party dependency this lesson touches is `python-dotenv`, and it is optional - every cell below is pure standard-library Python. The install line is commented out so the notebook runs as-is on a machine that already has what it needs; uncomment it on a fresh Colab or clean environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A one-line environment-prep cell - it installs nothing by default.

**How the code works - step by step**
- **`# !pip install python-dotenv -q`** - commented out; uncomment only on Colab or a fresh virtualenv to add `.env` file support. The `# noqa` silences linters on the shell-magic line.

**In one line:** optional dependency, off by default.

**What you'll see:** (no output - environment prep)

## Setup - API keys (optional here)

This cell reserves the three provider key slots the wider Module 9 uses, but nothing in *this* notebook actually calls a model - every cell is local arithmetic. You can run the whole lesson with these left blank; the keys only matter once you reach the hands-on fine-tuning notebooks (9.2 onward).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not a model call - it seeds empty placeholders for OpenAI, Anthropic, and Google keys so later lessons can find them, and deliberately avoids hardcoding secrets.

**How the code works - step by step**
- **`import os`** - standard library, used only to reach `os.environ`.
- **`os.environ.setdefault(...)`** (x3) - sets each key to an empty string *only if it is not already defined*, so a real key you exported in your shell is never overwritten.
- The trailing comments point to where each key comes from (platform.openai.com, console.anthropic.com, aistudio.google.com).

**In one line:** reserve the key slots without clobbering real ones - no key is required to run this notebook.

**What you'll see:** (no output - environment prep)

## Reproducibility marker

A single-comment cell flagging that the lesson leans on seeded/deterministic values so your numbers match the walkthroughs exactly.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A no-op placeholder cell - it documents intent and runs nothing.

**How the code works - step by step**
- The cell body is a lone comment noting the lesson uses seeded random values throughout, so outputs are stable across runs.

**In one line:** a note, not code.

**What you'll see:** No output - it is a comment-only cell.

## 1 - Three doors: prompt, RAG, or fine-tune?

Every fine-tuning decision starts by naming the *gap*: is the model missing **knowledge** (facts it does not have) or is its **behavior** wrong (right facts, wrong format/tone)? A knowledge gap is a RAG job; a behavior gap that survives good prompting is a fine-tuning job - but only with a baseline and curated data. This cell encodes that diagnosis as one small classifier so you can watch which door four everyday cases land on.

In [ ]:
# THE DECISION: prompt, RAG, or fine-tune? Diagnose the GAP first.
def recommend(new_or_changing_knowledge, behavior_gap, prompt_baseline_done,
              have_examples, calls_per_day=0):
    # a KNOWLEDGE gap -> RAG (never bake changing facts into weights)
    if new_or_changing_knowledge:
        return ("RAG", "Knowledge gap: retrieve it, do not bake it into weights.")
    # never skip the cheap rung
    if not prompt_baseline_done:
        return ("PROMPT", "No baseline yet: try system prompts, few-shot, structured outputs first.")
    # a BEHAVIOR gap that survived prompting -> fine-tune IF you have data
    if behavior_gap and have_examples:
        if calls_per_day >= 10000:
            return ("FINE-TUNE", "Behavior gap at volume: fine-tune to lock format and cut per-call cost.")
        return ("FINE-TUNE", "Behavior gap + curated data: fine-tune to change how it responds.")
    if behavior_gap and not have_examples:
        return ("PROMPT", "Behavior gap but no data: few-shot + a stronger system prompt first.")
    return ("PROMPT", "Prompting covers most cases; nothing here forces an escalation.")

cases = [
    ("Support bot, docs change weekly",     dict(new_or_changing_knowledge=True,  behavior_gap=False, prompt_baseline_done=True,  have_examples=False)),
    ("Clinical-note formatter, docs static",dict(new_or_changing_knowledge=False, behavior_gap=True,  prompt_baseline_done=True,  have_examples=True, calls_per_day=40000)),
    ("Legal summarizer, first attempt",     dict(new_or_changing_knowledge=False, behavior_gap=True,  prompt_baseline_done=False, have_examples=False)),
    ("Tone match, no dataset yet",          dict(new_or_changing_knowledge=False, behavior_gap=True,  prompt_baseline_done=True,  have_examples=False)),
]
for name, p in cases:
    rec, why = recommend(**p)
    print(f"  {name:38s} -> {rec:11s} | {why}")

# Output:
#   Support bot, docs change weekly        -> RAG         | Knowledge gap: retrieve it, do not bake it into weights.
#   Clinical-note formatter, docs static   -> FINE-TUNE   | Behavior gap at volume: fine-tune to lock format and cut per-call cost.
#   Legal summarizer, first attempt        -> PROMPT      | No baseline yet: try system prompts, few-shot, structured outputs first.
#   Tone match, no dataset yet             -> PROMPT      | Behavior gap but no data: few-shot + a stronger system prompt first.

**Explanation**

This is a decision function, not a model - `recommend` reads five facts about your use case and returns a route plus a one-line reason. The whole point is the *order* of its checks: knowledge short-circuits to RAG, a missing baseline short-circuits to PROMPT, and only a behavior gap that has data reaches FINE-TUNE.

**How the code works - step by step**
- **`recommend(...)`** - takes the two gap questions (`new_or_changing_knowledge`, `behavior_gap`) plus whether a `prompt_baseline_done`, whether you `have_examples`, and `calls_per_day`.
- **First guard** - any knowledge gap returns `RAG` immediately (never bake changing facts into weights).
- **Second guard** - no baseline yet returns `PROMPT` (try system prompts, few-shot, structured outputs first).
- **Behavior + data** - returns `FINE-TUNE`; if `calls_per_day >= 10000` the reason upgrades to the volume case.
- **Behavior but no data** - falls back to `PROMPT` (few-shot + stronger system prompt).
- **`cases`** - four scenarios (support bot, clinical formatter, legal summarizer, tone match) run through the classifier in a loop.

**In one line:** knowledge -> RAG, no baseline -> PROMPT, behavior + data -> FINE-TUNE.

**What you'll see:** Four aligned rows: the support bot (changing docs) -> RAG, the clinical-note formatter (behavior gap, has data, 40k/day) -> FINE-TUNE, the legal summarizer (no baseline) -> PROMPT, and the tone-match case (behavior gap, no dataset) -> PROMPT. Only one of the four lands on FINE-TUNE.

## 2 - The escalation ladder: climb only as high as you must

The five techniques form a ladder - prompt, few-shot, RAG, fine-tune, pre-train - and each rung costs far more than the one below. The discipline is to start at the bottom and stop the instant your problem is solved. This cell prints the ladder and, crucially, refuses to print rungs above the one that already solves your case.

In [ ]:
# THE ESCALATION LADDER: climb only as high as your problem needs.
LADDER = [
    ("1 Prompt engineering", "hours",   "~$0",        "model already has the knowledge/skill"),
    ("2 Few-shot",           "hours",   "~$0",        "a few examples pin the format"),
    ("3 RAG",                "2-4 wks", "$/mo infra", "the gap is KNOWLEDGE the model lacks"),
    ("4 Fine-tune",          "1-8 wks", "$$ + data",  "BEHAVIOR must be reliable at volume"),
    ("5 Pre-train",          "months",  "$$$$$",      "almost never - you are building a base model"),
]
def climb_to(rung_that_solves_it):
    print(f"  Target solved at rung {rung_that_solves_it}. Climb only this far:\n")
    print(f"  {'Rung':22s} {'Time':8s} {'Cost':12s} Stop-here-when")
    print("  " + "-"*74)
    for i, (name, t, c, stop) in enumerate(LADDER, 1):
        mark = "  <= STOP" if i == rung_that_solves_it else ""
        print(f"  {name:22s} {t:8s} {c:12s} {stop}{mark}")
        if i == rung_that_solves_it:
            break

climb_to(3)   # a knowledge problem is solved at RAG; do not climb to fine-tune

# Output:
#   Target solved at rung 3. Climb only this far:
#
#   Rung                   Time     Cost         Stop-here-when
#   --------------------------------------------------------------------------
#   1 Prompt engineering   hours    ~$0          model already has the knowledge/skill
#   2 Few-shot             hours    ~$0          a few examples pin the format
#   3 RAG                  2-4 wks  $/mo infra   the gap is KNOWLEDGE the model lacks  <= STOP

**Explanation**

A reference table plus a printer, not a model call. `LADDER` is the cost/time cheat sheet; `climb_to` is a visual discipline device - it stops printing at the rung that clears your ceiling, so you literally cannot see (or reach for) the more expensive rungs above it.

**How the code works - step by step**
- **`LADDER`** - a five-row list of `(rung name, time, cost, stop-here-when)` from prompt engineering (hours, ~$0) up to pre-training (months, $$$$$).
- **`climb_to(rung_that_solves_it)`** - prints a header, then loops the rungs with `enumerate(..., 1)`, marking the solving rung with `<= STOP` and **breaking** the loop right after it.
- **`climb_to(3)`** - called for a knowledge problem solved at RAG, so rungs 4 (fine-tune) and 5 (pre-train) are never printed.

**In one line:** list the rungs cheapest-first, then stop the moment one clears your ceiling.

**What you'll see:** A three-row table (rungs 1-3) ending with RAG flagged `<= STOP`; fine-tune and pre-train never appear, reinforcing that a knowledge problem should not climb past RAG.

## 3 - Cost and break-even: when fine-tuning pays back

Fine-tuning's economics are the mirror image of prompting's: prompting has zero setup but a higher forever per-call cost (long prompts, retrieved context), while fine-tuning pays a big up-front cost and then runs on short prompts. Whether that trade wins is pure arithmetic. This cell sweeps three call volumes over six months and prints which method is cheapest at each.

In [ ]:
# COST OVER 6 MONTHS: prompt vs RAG vs fine-tune. Prices are ILLUSTRATIVE
# (Jun 2026 $/1M tokens) - always verify current rates.
PRICE = {"gemini-2.5-flash": (0.30, 2.50), "gpt-4.1-mini": (0.80, 3.20)}

def costs(calls_per_day, tin=500, tout=200, months=6, model="gemini-2.5-flash"):
    pin, pout = PRICE[model]
    days = months * 30
    def per_call(extra_in):
        return ((tin + extra_in)*pin + tout*pout) / 1e6
    prompt = per_call(200) * calls_per_day * days                    # system-prompt overhead
    rag    = per_call(800) * calls_per_day * days + 50*months + 200  # + context, vector DB, indexing
    ft     = per_call(0)   * calls_per_day * days + 50 + 500         # short prompts + train + data prep
    return {"Prompting": prompt, "RAG": rag, "Fine-tune": ft}

for cpd in (100, 5000, 100000):
    c = costs(cpd)
    win = min(c, key=c.get)
    print(f"  {cpd:>7,}/day 6-mo:  " + "  ".join(f"{k} ${v:,.0f}" for k, v in c.items()) + f"   -> {win}")

# Output:
#         100/day 6-mo:  Prompting $13  RAG $516  Fine-tune $562   -> Prompting
#       5,000/day 6-mo:  Prompting $639  RAG $1,301  Fine-tune $1,135   -> Prompting
#     100,000/day 6-mo:  Prompting $12,780  RAG $16,520  Fine-tune $12,250   -> Fine-tune

**Explanation**

A cost calculator, not a model call - it converts token overhead and setup fees into dollars and picks the minimum. The core idea it makes concrete: fine-tuning trades a large fixed cost for a smaller variable cost, so the winner flips only once volume is high enough to amortize the setup.

**How the code works - step by step**
- **`PRICE`** - illustrative Jun-2026 `$/1M` (input, output) rates for two small models.
- **`per_call(extra_in)`** - inner helper: cost of one call given each method's extra input tokens.
- **`prompt`** - adds +200 input tokens of system-prompt overhead per call, no setup.
- **`rag`** - adds +800 tokens of retrieved context per call, plus monthly vector-DB + indexing fees.
- **`ft`** - zero per-call overhead (short prompts) but a one-time +50 training +500 data-prep charge.
- **Sweep loop** - runs `costs(cpd)` at 100, 5,000, and 100,000 calls/day and prints the `min` as the winner.

**In one line:** short prompts make fine-tune cheaper per call, but only volume repays its setup.

**What you'll see:** Three lines: at 100/day Prompting wins ($13 vs $516 vs $562), at 5,000/day Prompting still wins ($639 vs $1,301 vs $1,135), and only at 100,000/day does Fine-tune become cheapest ($12,250 vs $12,780 vs $16,520). Prices are illustrative - verify current rates.

## 4 - The method taxonomy: SFT, DPO, GRPO, KTO, distillation

"Fine-tuning" is not one thing. Once you have decided to do it, a second decision appears - *which* method - and it is driven by your goal (teach a format? shift tone? improve reasoning? cut cost?) and the shape of data you can actually get. This cell is a lookup that routes goal-and-data to the right objective.

In [ ]:
# METHOD TAXONOMY: match the METHOD to the GOAL + the DATA you have.
def pick_method(goal, data):
    table = {
        ("format",    "pairs"):       ("SFT",          "input->output pairs teach the task format"),
        ("style",     "preferences"): ("DPO",          "chosen/rejected pairs shape tone - the 2026 default"),
        ("style",     "thumbs"):      ("KTO",          "unpaired up/down feedback, no matched pairs needed"),
        ("reasoning", "verifiable"):  ("GRPO",         "sample many, reward the correct ones (math/code)"),
        ("cost",      "teacher"):     ("Distillation", "large teacher -> small student for cheap inference"),
        ("new-domain","corpus"):      ("Continual PT", "large unlabeled corpus adds domain fluency"),
    }
    return table.get((goal, data), ("SFT", "default: start with supervised fine-tuning"))

for goal, data in [("format","pairs"), ("style","preferences"), ("reasoning","verifiable"),
                   ("cost","teacher"), ("style","thumbs"), ("new-domain","corpus")]:
    m, why = pick_method(goal, data)
    print(f"  goal={goal:10s} data={data:12s} -> {m:13s} | {why}")

# Output:
#   goal=format     data=pairs        -> SFT           | input->output pairs teach the task format
#   goal=style      data=preferences  -> DPO           | chosen/rejected pairs shape tone - the 2026 default
#   goal=reasoning  data=verifiable   -> GRPO          | sample many, reward the correct ones (math/code)
#   goal=cost       data=teacher      -> Distillation  | large teacher -> small student for cheap inference
#   goal=style      data=thumbs       -> KTO           | unpaired up/down feedback, no matched pairs needed
#   goal=new-domain data=corpus       -> Continual PT  | large unlabeled corpus adds domain fluency

**Explanation**

A dispatch table, not a model call - `pick_method` keys on the `(goal, data)` pair and returns the matching method plus its reason. The lesson is that the *same* goal routes differently depending on the data you have (preference pairs vs thumbs vs verifiable rewards), so the data shape, not the goal alone, picks the method.

**How the code works - step by step**
- **`pick_method(goal, data)`** - builds a dict `table` mapping six `(goal, data)` tuples to `(method, reason)`.
- **The six rows** - format+pairs -> SFT, style+preferences -> DPO, style+thumbs -> KTO, reasoning+verifiable -> GRPO, cost+teacher -> Distillation, new-domain+corpus -> Continual PT.
- **`table.get((goal, data), default)`** - unknown combinations fall back to SFT.
- **Loop** - runs six representative pairs and prints the routed method and its one-line rationale.

**In one line:** the (goal, data) pair - not the goal alone - selects the objective.

**What you'll see:** Six aligned rows mapping each goal/data pair to its method: format/pairs -> SFT, style/preferences -> DPO, reasoning/verifiable -> GRPO, cost/teacher -> Distillation, style/thumbs -> KTO, new-domain/corpus -> Continual PT, each with a short reason.

## 5 - Data: quality crushes quantity

The instinct is "more data is better"; for fine-tuning it is often the opposite. Because fine-tuning re-shapes behavior rather than installing knowledge, a small set of near-perfect examples teaches the target better than a large noisy one - the noise actively teaches bad habits (the LIMA / AlpaGasus result). This cell reproduces that effect in miniature by scoring a raw corpus against a filtered subset.

In [ ]:
# QUALITY CRUSHES QUANTITY: a filtered SMALL set beats a big NOISY one (LIMA).
raw = [{"id": i, "quality": 0.95 if i % 4 == 0 else 0.35} for i in range(2000)]

def train_score(examples):
    if not examples:
        return 0.0
    avg_q = sum(e["quality"] for e in examples) / len(examples)
    coverage = min(len(examples) / 1000, 1.0)             # returns diminish past ~1000
    return round(100 * (0.55*avg_q + 0.45*coverage*avg_q), 1)   # quality GATES coverage

filtered = [e for e in raw if e["quality"] >= 0.9][:1000]     # keep only clean examples
print(f"  RAW  : {len(raw):5d} examples, avg quality {sum(e['quality'] for e in raw)/len(raw):.2f} -> score {train_score(raw)}")
print(f"  CLEAN: {len(filtered):5d} examples, avg quality {sum(e['quality'] for e in filtered)/len(filtered):.2f} -> score {train_score(filtered)}")
print("  Fewer, cleaner examples win: curation beats accumulation (LIMA/AlpaGasus).")

# Output:
#   RAW  :  2000 examples, avg quality 0.50 -> score 50.0
#   CLEAN:   500 examples, avg quality 0.95 -> score 73.6
#   Fewer, cleaner examples win: curation beats accumulation (LIMA/AlpaGasus).

**Explanation**

A scoring demo, not a model call - it models a training outcome with a formula where **quality gates coverage**, so noise cannot be averaged away by adding more of it. Filtering a big noisy set down to a small clean one raises the score even though the count drops.

**How the code works - step by step**
- **`raw`** - 2,000 synthetic examples where every 4th has quality 0.95 and the rest 0.35 (mostly noise).
- **`train_score(examples)`** - averages quality, computes `coverage` that saturates at 1,000 examples, then combines them so `avg_q` multiplies the coverage term - quality *gates* the benefit of volume.
- **`filtered`** - keeps only examples with quality >= 0.9, capped at 1,000 (here that yields 500 clean ones).
- **Two prints** - score the raw set and the clean subset side by side.

**In one line:** filter to the clean examples and the smaller set scores higher - curation beats accumulation.

**What you'll see:** Two lines: RAW 2000 examples (avg quality 0.50) scores 50.0, while CLEAN 500 examples (avg quality 0.95) scores 73.6 - a quarter of the data, a much higher score - plus a one-line takeaway citing LIMA/AlpaGasus.

## 6 - PEFT economics: LoRA, QLoRA, and full

Fine-tuning sounds like it needs a data center, but you rarely update all the weights. Parameter-efficient fine-tuning trains a tiny set of adapter weights and freezes the rest - LoRA at ~95% of full quality, QLoRA adding 4-bit quantization so a 7B model fits on a free Colab T4. This cell estimates the training VRAM for each method and reports which GPU it fits on.

In [ ]:
# PEFT ECONOMICS: how much VRAM to TRAIN a 7B, and where it fits.
def peft_plan(params_b, method):
    # rough VRAM (GB) to train, by method (full needs optimizer states -> ~16 bytes/param)
    vram = {"full": params_b*16, "lora": params_b*3.4, "qlora": params_b*1.3, "dora": params_b*1.4}[method]
    quality = {"full": 1.00, "lora": 0.95, "qlora": 0.94, "dora": 0.96}[method]
    fits_t4, fits_4090, fits_a100 = vram <= 16, vram <= 24, vram <= 80
    where = ("free T4 (16GB)" if fits_t4 else "one RTX-4090 (24GB)" if fits_4090
             else "one A100/H100 (80GB)" if fits_a100 else "multi-GPU A100/H100")
    return vram, where, quality

print("  7B model, VRAM to train + where it fits + ~quality vs full:")
for m in ("full", "lora", "qlora", "dora"):
    v, where, q = peft_plan(7, m)
    print(f"    {m:6s} ~{v:5.1f} GB  fits: {where:20s}  ~{q*100:.0f}% of full quality")

# Output:
#   7B model, VRAM to train + where it fits + ~quality vs full:
#     full   ~112.0 GB  fits: multi-GPU A100/H100   ~100% of full quality
#     lora   ~ 23.8 GB  fits: one RTX-4090 (24GB)   ~95% of full quality
#     qlora  ~  9.1 GB  fits: free T4 (16GB)        ~94% of full quality
#     dora   ~  9.8 GB  fits: free T4 (16GB)        ~96% of full quality

**Explanation**

A VRAM calculator, not a model call - it turns parameter count and method into a rough gigabyte figure and a hardware bucket. The point it makes concrete: raw quality falls full > DoRA/LoRA > QLoRA, but accessibility runs the exact reverse, and QLoRA's ~94% at ~$0 is the pragmatic default.

**How the code works - step by step**
- **`peft_plan(params_b, method)`** - looks up a per-method bytes-per-param factor: full ~16 (optimizer states dominate), lora ~3.4, qlora ~1.3, dora ~1.4, times the billions of params to get VRAM in GB.
- **`quality`** - a companion dict giving each method's rough fraction of full quality (full 1.00, lora 0.95, qlora 0.94, dora 0.96).
- **Fit checks** - three booleans (`<=16` free T4, `<=24` RTX-4090, `<=80` A100/H100) pick the smallest GPU the job fits on.
- **Loop** - runs a 7B model through all four methods and prints VRAM, GPU, and quality.

**In one line:** freeze most weights, train tiny adapters, and a 7B fine-tune drops from ~112GB to ~9GB.

**What you'll see:** Four rows for a 7B model: full ~112GB (multi-GPU, 100%), lora ~23.8GB (one RTX-4090, 95%), qlora ~9.1GB (free T4, 94%), dora ~9.8GB (free T4, 96%) - QLoRA and DoRA both fit a free 16GB T4.

## 7 - Catastrophic forgetting: the regression gate

Fine-tuning has a dangerous failure mode: optimizing hard for one task shifts the weights away from unrelated skills, so your support model gets great at support and quietly loses basic math or safety behavior. The only defense is a regression suite that scores the capabilities you want to *preserve*, run before and after. This cell is that gate as code.

In [ ]:
# CATASTROPHIC FORGETTING: a regression suite catches loss the task metric hides.
def regression(before, after, floor=0.05):
    print(f"  {'capability':12s} {'before':>7s} {'after':>7s}  verdict")
    print("  " + "-"*44)
    forgot = []
    for cap in before:
        b, a = before[cap], after[cap]
        drop = b - a
        verdict = "OK" if drop <= floor else f"FORGOT ({a-b:+.2f})"
        if drop > floor and cap != "support_task":
            forgot.append(cap)
        print(f"  {cap:12s} {b:7.2f} {a:7.2f}  {verdict}")
    print("  -> SHIP" if not forgot else f"  -> BLOCK: fine-tune degraded {forgot} (data-mix + lower LR + re-test)")

before = {"support_task": 0.62, "math": 0.71, "safety": 0.93, "general": 0.68}
after  = {"support_task": 0.94, "math": 0.44, "safety": 0.90, "general": 0.55}
regression(before, after)

# Output:
#   capability    before   after  verdict
#   --------------------------------------------
#   support_task    0.62    0.94  OK
#   math            0.71    0.44  FORGOT (-0.27)
#   safety          0.93    0.90  OK
#   general         0.68    0.55  FORGOT (-0.13)
#   -> BLOCK: fine-tune degraded ['math', 'general'] (data-mix + lower LR + re-test)

**Explanation**

A pass/fail gate, not a model call - `regression` diffs before/after scores across every capability (not just the trained task) and blocks the ship if anything unrelated regressed past a floor. It replaces the misleading "vibe check" on the target task with a real, full-coverage test.

**How the code works - step by step**
- **`regression(before, after, floor=0.05)`** - iterates the capabilities, computes `drop = before - after` for each.
- **Verdict** - a drop within `floor` prints `OK`; a larger drop prints `FORGOT` with the signed delta.
- **`forgot` list** - collects any non-`support_task` capability that dropped past the floor (the trained task is allowed to move).
- **Gate** - prints `SHIP` if `forgot` is empty, else `BLOCK` naming the degraded capabilities and the fix (data-mix + lower LR + re-test).
- **`before` / `after`** - sample scorecards where support jumps but math and general slip.

**In one line:** score everything before and after, and block the ship if any untrained skill regressed.

**What you'll see:** A four-row capability table: support_task 0.62 -> 0.94 (OK), math 0.71 -> 0.44 (FORGOT -0.27), safety 0.93 -> 0.90 (OK), general 0.68 -> 0.55 (FORGOT -0.13), ending in `-> BLOCK` because math and general regressed.

## 8 - India fine-tuning: tokenizer fertility beats model size

For Hindi and other Indic languages the counter-intuitive lesson is that tokenizer efficiency matters more than parameter count: an Indic-first tokenizer represents the same Hindi text in far fewer tokens, which means cheaper inference and a smaller model that punches above its weight. This cell prices the same Hindi workload under a generic tokenizer versus Sarvam-1's Indic tokenizer.

In [ ]:
# INDIA: an Indic-first tokenizer costs FEWER tokens on the same Hindi text.
def monthly_bill(chars, fertility, queries_per_month, price_in_per_1m=0.30):
    tokens_per_query = chars * fertility        # fertility = tokens per character (illustrative)
    bill = tokens_per_query * queries_per_month * price_in_per_1m / 1e6
    return bill, tokens_per_query

CHARS, QPM = 800, 500000
for name, fert in [("Llama-style tokenizer", 0.90), ("Sarvam-1 Indic tokenizer", 0.38)]:
    bill, tpq = monthly_bill(CHARS, fert, QPM)
    print(f"  {name:26s}: {tpq:6.0f} tok/query  ~${bill:6.0f}/mo input at {QPM:,} queries")
print("  Indic-first tokenizer ~2x fewer tokens on the same Hindi text -> ~half the token bill.")

# Output:
#   Llama-style tokenizer     :    720 tok/query  ~$   108/mo input at 500,000 queries
#   Sarvam-1 Indic tokenizer  :    304 tok/query  ~$    46/mo input at 500,000 queries
#   Indic-first tokenizer ~2x fewer tokens on the same Hindi text -> ~half the token bill.

**Explanation**

A billing calculator, not a model call - it multiplies characters by a fertility factor (tokens per character) to get tokens per query, then a monthly input bill. It isolates one variable, tokenizer fertility, to show the cost gap exists *before* any fine-tuning happens.

**How the code works - step by step**
- **`monthly_bill(chars, fertility, queries_per_month, price_in_per_1m)`** - `tokens_per_query = chars * fertility`, then scales by monthly volume and the per-1M input price.
- **`CHARS, QPM`** - a fixed 800-character query at 500,000 queries/month for both tokenizers.
- **Loop** - compares a Llama-style tokenizer (fertility 0.90) against Sarvam-1's Indic tokenizer (fertility 0.38) on identical text.
- **Print** - tokens per query and the monthly bill for each, plus a takeaway.

**In one line:** same Hindi text, half the tokens under an Indic-first tokenizer, so half the bill.

**What you'll see:** Two lines: the Llama-style tokenizer costs 720 tok/query (~$108/mo) while the Sarvam-1 Indic tokenizer costs 304 tok/query (~$46/mo) on the same Hindi text - roughly half the tokens and half the bill before any fine-tuning.

Across eight small programs you built the entire fine-tune decision as executable logic: name the gap (knowledge vs behavior), climb the escalation ladder only as high as you must, check whether volume amortizes the setup cost, match the method to your goal-and-data, curate rather than accumulate, size the job to fit a modest GPU with QLoRA, and guard the result with a regression suite before shipping. The through-line every cell reinforces: fine-tuning changes behavior, not knowledge, and it is the last rung you climb, not the first. Lesson 9.2 picks this up hands-on by building the curated dataset this notebook kept insisting matters most.